# 09 · Import the Actual Hackathon Course Materials

**Run All** in VS Code after installing `requirements.txt`. The PDF documents in `Clinical.zip` and `FoodSecurity.zip` are indexed as real source documents with page citations. Large ClinicalTrials CSVs are cataloged for structured analysis, not injected as unverified free-text evidence. The Cisco, Agilent and Google TXT files contain **links**, not full documents; we save them in `source_catalog.json` as `LINK_ONLY_NOT_DOWNLOADED`. The original content must be fetched successfully before its technical claims can be used as RAG evidence.

*Archive ingestion may take a minute or two for 600+ pages of FAO PDFs. Re-running is safe because identical documents are deduplicated.*

In [1]:
%run 01_document_ingestion.ipynb


ModuleNotFoundError: No module named 'nbformat'

In [ ]:
# Real course-data intake: PDFs are evidence; URL lists are only discovery catalogs.
from pathlib import PurePosixPath
from urllib.parse import urlparse
from collections import Counter

COURSE = ROOT / 'data' / 'course_sources'
COURSE.mkdir(parents=True, exist_ok=True)
SOURCES = {
    'Agilent.txt': 'clinical',
    'DataSource-Cisco.txt': 'technology',
    'Google GCP.txt': 'technology',
}
ARCHIVES = {'Clinical.zip': 'clinical', 'FoodSecurity.zip': 'csr'}
MANIFEST_PATH = COURSE / 'import_manifest.json'
CATALOG_PATH = COURSE / 'source_catalog.json'

def catalog_link_files():
    catalog=[]
    for filename,domain in SOURCES.items():
        filepath=COURSE/filename
        if not filepath.exists():
            print('Missing:',filename)
            continue
        lines=[l.strip() for l in filepath.read_text(encoding='utf-8-sig',errors='replace').splitlines()]
        for i,line in enumerate(lines):
            urls=re.findall(r'https?://[^\s<>\"\']+',line)
            for url in urls:
                url=url.rstrip(').,;')
                if urlparse(url).scheme!='https':
                    continue
                label=next((x.strip(' 0123456789.:-') for x in reversed(lines[max(0,i-3):i])
                            if x and not x.startswith(('https://','http://'))),filename)
                catalog.append({'domain':domain,'directory_file':filename,'title_hint':label,
                                'url':url,'status':'LINK_ONLY_NOT_DOWNLOADED',
                                'note':'URL listing is not the content of the underlying document.'})
    CATALOG_PATH.write_text(json.dumps(catalog,indent=2,ensure_ascii=False))
    return catalog

def ingest_course_archives():
    # Extract directly from ZIP bytes, no filesystem extraction or zip-slip risks.
    # CSVs are large clinical-trials datasets, not pages to treat as PDF evidence.
    imported=[]; csv_inventory=[]; failures=[]
    for filename,domain in ARCHIVES.items():
        path=COURSE/filename
        if not path.exists():
            failures.append({'archive':filename,'reason':'Not present'});continue
        with zipfile.ZipFile(path) as archive:
            members=archive.infolist()
            if len(members)>200: raise ValueError('Too many archive entries')
            total=sum(x.file_size for x in members if not x.is_dir())
            if total>256*1024*1024:raise ValueError('Archive decompressed size over 256 MiB')
            for member in members:
                if member.is_dir():continue
                name=member.filename; rel=PurePosixPath(name)
                if name.startswith('/') or '..' in rel.parts or '\\' in name or (member.flag_bits & 1):
                    failures.append({'archive':filename,'member':name,'reason':'Unsafe path or encrypted member'});continue
                ext=rel.suffix.lower()
                if ext=='.csv':
                    csv_inventory.append({'archive':filename,'member':name,'size_bytes':member.file_size,
                                          'status':'CATALOG_ONLY','note':'Large dataset; preview in notebook 11; not indexed as natural-language evidence'})
                    continue
                if ext!='.pdf':continue
                if member.file_size>MAX_UPLOAD:
                    failures.append({'archive':filename,'member':name,'reason':'PDF exceeds maximum document size'});continue
                # Prevent decompression-bomb ratios; normal PDFs are already compressed.
                if member.file_size>10*1024*1024 and member.compress_size>0 and member.file_size/member.compress_size>200:
                    failures.append({'archive':filename,'member':name,'reason':'Suspicious compression ratio'});continue
                # Preserve document path in import manifest while using safe short filename in the citations.
                safe_name=rel.name
                if len(safe_name)>170:
                    stem=Path(safe_name).stem[:150]
                    safe_name=stem+'.pdf'
                try:
                    result=ingest_blob(PROJECT_ID,domain,safe_name,archive.read(member),source_type='course-archive')
                    imported.append({'archive':filename,'member':name,'domain':domain,
                                     'citation_filename':safe_name,**result})
                except Exception as exc:
                    failures.append({'archive':filename,'member':name,'reason':str(exc)[:300]})
    manifest={'project_id':PROJECT_ID,'imported':imported,'csv_inventory':csv_inventory,'failures':failures,
              'important_note':'TXT sources are URL directories; SEC filings and IBM technical documents were not supplied.'}
    MANIFEST_PATH.write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
    return manifest

catalog=catalog_link_files()
manifest=ingest_course_archives()
summary={'official_source_links_catalogued':len(catalog),
         'pdfs_imported_or_skipped_as_duplicates':len(manifest['imported']),
         'pdfs_newly_indexed':sum(not row['duplicate'] for row in manifest['imported']),
         'pdf_chunks_newly_indexed':sum(row.get('chunks',0) for row in manifest['imported']),
         'clinical_csv_datasets_catalogued':len(manifest['csv_inventory']),
         'failures':manifest['failures']}
print(json.dumps(summary,indent=2))
assert len(manifest['imported'])>0,'No actual course PDFs imported: check data/course_sources'


## What's still missing?
- **Technology evidence:** Source URLs supplied; official PDF/web contents must be fetched separately using notebook 12.
- **Agilent instrument evidence:** Product page URLs supplied; actual documentation needs fetching.
- **SEC:** Cisco/Agilent 2025 10-K reports not supplied.
- **Clinical:** The archive contains scientific papers on COVID, diabetes, heart attack, and knee injuries, **not** Agilent instrument service manuals.
- **CSR:** Both UN SOFI reports are now actually indexed, with page-level citations.

Next run `10_course_rag_validation.ipynb`, then optionally `11_clinical_dataset_preview.ipynb`.